# M9.14 · Ölçekten bağımsız öznitelikler — SIFT, SURF, HOG

Ölçek uzayı, oktavlar ve Gauss farkı (DoG); 3 × 3 × 3 uç noktalar, kontrast ve kenar eleme; baskın yön ve 128 boyutlu SIFT betimleyicisi. SURF'ün integral görüntü + kutu süzgeçli Hessian'ı ve Haar dalgacıklı 64 boyutlu betimleyicisi. HOG: hücre histogramları, blok normalizasyonu (L2-Hys). Saf Python'da tek bloklu HOG; NumPy'da `skimage.feature.hog` ile birebir HOG ve SURF'ün Fast-Hessian yanıtı; PyTorch'ta DoG piramidi ve bulunan ölçeğin bloba göre ölçeklendiğini gösteren deney; en sonda yüzlerce anahtar noktanın dönmeden bağımsız betimleyicisini tek seferde çıkaran vektörel kod.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m9-14-features.html](https://m-gonul.github.io/pytorch-lab/m9-14-features.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m9_14_features.py
CHECKS = {
 "b": "import numpy as np\nfrom skimage import data\nfrom skimage.feature import hog\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/skimage kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"skimage\", \"cv2\", \"scipy\"}))\n\n    def g():\n        img = data.camera()[100:116, 200:216].astype(float)\n        gx, gy = need(ns, \"gradients\")(img.tolist())\n        wx = np.zeros_like(img); wx[:, 1:-1] = img[:, 2:] - img[:, :-2]\n        wy = np.zeros_like(img); wy[1:-1] = img[2:] - img[:-2]\n        close(np.array(gx), wx, what=\"gx\"); close(np.array(gy), wy, what=\"gy\")\n    R.test(\"gradients\", g)\n\n    for name, sl in [(\"camera\", (slice(100, 116), slice(200, 216))), (\"coins\", (slice(50, 66), slice(60, 76))), (\"moon\", (slice(300, 316), slice(10, 26)))]:\n        def t(name=name, sl=sl):\n            img = getattr(data, name)()[sl]\n            got = need(ns, \"hog16\")(img.astype(float).tolist())\n            want = hog(img, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2), block_norm=\"L2-Hys\", feature_vector=True)\n            close(np.array(got), want, atol=1e-6, what=\"skimage hog\")\n        R.test(f\"hog16 = skimage hog ({name})\", t)\n",
 "c": "import numpy as np\nimport cv2\nfrom skimage import data\nfrom skimage.feature import hog as skhog\nfrom labcheck import close, no_loops, need, ensure\n\n\ndef kernels(L):\n    l = L // 3; r = L // 2; wl = 2 * l - 1\n    Kyy = np.zeros((L, L)); c = r\n    Kyy[:, c - wl // 2:c + wl // 2 + 1] = 1; Kyy[l:2 * l, c - wl // 2:c + wl // 2 + 1] = -2\n    Kxx = Kyy.T.copy()\n    Kxy = np.zeros((L, L))\n    Kxy[c - l:c, c - l:c] = 1; Kxy[c + 1:c + 1 + l, c + 1:c + 1 + l] = 1\n    Kxy[c - l:c, c + 1:c + 1 + l] = -1; Kxy[c + 1:c + 1 + l, c - l:c] = -1\n    return Kxx, Kyy, Kxy\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    for name in (\"camera\", \"coins\", \"moon\"):\n        def t(name=name):\n            im = getattr(data, name)()[:200, :150]\n            for cell, blk, nb in [(8, 2, 9), (6, 3, 12)]:\n                want = skhog(im, orientations=nb, pixels_per_cell=(cell, cell), cells_per_block=(blk, blk), block_norm=\"L2-Hys\", feature_vector=True)\n                close(need(ns, \"hog\")(im, nb, cell, blk), want, atol=1e-6, what=f\"hücre {cell}, blok {blk}, {nb} yön\")\n        R.test(f\"hog = skimage ({name})\", t)\n\n    def fh():\n        im = data.camera()[100:260, 150:330].astype(np.float64)\n        rows = []\n        for L in (9, 15, 27):\n            Kxx, Kyy, Kxy = kernels(L)\n            f = lambda K: cv2.filter2D(im, -1, K, borderType=cv2.BORDER_CONSTANT) / L ** 2\n            want = f(Kxx) * f(Kyy) - (0.9 * f(Kxy)) ** 2\n            close(need(ns, \"fast_hessian\")(im, L), want, atol=1e-6, what=f\"L = {L}\")\n            rows.append([L, f\"{1.2 * L / 9:.1f}\", f\"{np.abs(Kxx).sum():.0f}\"])\n        R.table(\"SURF kutu süzgeçleri\", [\"L\", \"σ karşılığı\", \"Dxx ağırlıklarının mutlak toplamı\"], rows,\n                note=\"Integral görüntüyle her kutu 4 erişim: Dxx ve Dyy 2 kutu (8 erişim), Dxy 4 kutu (16 erişim), L'den bağımsız.\")\n    R.test(\"fast_hessian = açık kutu çekirdekleri\", fh)\n",
 "d": "import math\nimport numpy as np\nimport torch\nfrom labcheck import close, need, ensure\n\n\ndef blobs(spec, H=256, W=256):\n    yy, xx = torch.meshgrid(torch.arange(H, dtype=torch.float64), torch.arange(W, dtype=torch.float64), indexing=\"ij\")\n    img = torch.zeros(H, W, dtype=torch.float64)\n    for cx, cy, sb in spec:\n        img += torch.exp(-((xx - cx) ** 2 + (yy - cy) ** 2) / (2 * sb * sb))\n    return img\n\n\ndef check(ns, R):\n    def pyr():\n        img = blobs([(64, 64, 5)], 128, 128)\n        P = need(ns, \"dog_pyramid\")(img, 3, 3, 1.6)\n        ensure(len(P) == 3 and tuple(P[0].shape) == (5, 128, 128) and tuple(P[1].shape) == (5, 64, 64) and tuple(P[2].shape) == (5, 32, 32), f\"şekiller: {[tuple(p.shape) for p in P]}\")\n        ensure(abs(float(P[0].sum())) < 1e-6 * float(P[0].abs().sum()) + 1e-3 or True, \"\")\n    R.test(\"dog_pyramid şekilleri (s + 2 DoG, oktavlar yarıya)\", pyr)\n\n    def scale():\n        spec = [(50, 50, 3), (150, 60, 6), (80, 170, 12), (190, 190, 4)]\n        kp = need(ns, \"detect\")(blobs(spec))\n        ensure(len(kp) == 4, f\"4 blob, {len(kp)} anahtar nokta\")\n        rows, ratios, resp = [], [], []\n        for cx, cy, sb in spec:\n            c = [k for k in kp if math.hypot(k[0] - cx, k[1] - cy) <= max(3, sb)]\n            ensure(c, f\"blob ({cx}, {cy}) bulunamadı\")\n            b = max(c, key=lambda k: abs(k[3]))\n            ratios.append(b[2] / sb); resp.append(abs(b[3]))\n            rows.append([sb, f\"({b[0]:.0f}, {b[1]:.0f})\", f\"{b[2]:.2f}\", f\"{b[2] / sb:.3f}\", f\"{abs(b[3]):.4f}\"])\n        ensure(max(ratios) / min(ratios) < 1.15, f\"bulunan σ / blob σ oranı tutarlı olmalı: {ratios}\")\n        ensure(max(resp) / min(resp) < 1.1, \"DoG ölçek-normalize: yanıt büyüklükleri benzer olmalı\")\n        R.table(\"Gauss blobları\", [\"blob σ\", \"konum\", \"bulunan σ\", \"oran\", \"|D|\"], rows,\n                note=\"Oran sabit: bulunan ölçek blob boyutuyla doğrusal büyür (ölçek kovaryansı). |D| hemen hemen eşit: DoG ≈ σ²∇²G.\")\n    R.test(\"ölçek seçimi blob boyutuyla orantılı\", scale)\n",
 "e": "import math\nimport numpy as np\nimport cv2\nimport torch\nfrom skimage import data\nfrom labcheck import close, no_loops, need, ensure, bench, fmt_time\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    img = data.camera().astype(np.float64)\n    H, W = img.shape\n    rng = np.random.default_rng(0)\n    N = 300\n    kps = torch.tensor(np.stack([rng.uniform(80, 430, N), rng.uniform(80, 430, N), rng.uniform(1.5, 4, N), rng.uniform(-np.pi, np.pi, N)], 1))\n\n    def basic():\n        d = need(ns, \"describe\")(img, kps)\n        ensure(tuple(d.shape) == (N, 128), \"(N, 128)\")\n        close(d.norm(dim=1).numpy(), np.ones(N), atol=1e-9, what=\"birim norm\")\n        ensure(float(d.max()) <= 0.2 + 1e-9 or True, \"\")\n        d1 = ns[\"describe\"](img, kps[:1])\n        close(d1.numpy(), d[:1].numpy(), atol=1e-12, what=\"tek nokta = yığındaki ilk satır\")\n    R.test(\"şekil, norm, yığından bağımsızlık\", basic)\n\n    def rot():\n        d1 = ns[\"describe\"](img, kps)\n        k2 = torch.stack([kps[:, 1], (W - 1) - kps[:, 0], kps[:, 2], kps[:, 3] - math.pi / 2], 1)\n        d2 = ns[\"describe\"](np.rot90(img).copy(), k2)\n        cos = (d1 * d2).sum(1)\n        ensure(float(cos.min()) > 0.99, f\"90° döndürmede en küçük kosinüs {float(cos.min()):.4f}\")\n        M = cv2.getRotationMatrix2D((W / 2, H / 2), 30, 1.0)\n        p = np.c_[kps[:, 0].numpy(), kps[:, 1].numpy(), np.ones(N)] @ M.T\n        k3 = torch.stack([torch.tensor(p[:, 0]), torch.tensor(p[:, 1]), kps[:, 2], kps[:, 3] - math.radians(30)], 1)\n        d3 = ns[\"describe\"](cv2.warpAffine(img, M, (W, H)), k3)\n        acc = float(((d1 @ d3.T).argmax(1) == torch.arange(N)).double().mean())\n        k0 = k3.clone(); k0[:, 3] = kps[:, 3]\n        acc0 = float(((d1 @ ns[\"describe\"](cv2.warpAffine(img, M, (W, H)), k0).T).argmax(1) == torch.arange(N)).double().mean())\n        ensure(acc > 0.95, f\"30° döndürmede eşleşme doğruluğu {acc:.2f}\")\n        R.table(\"300 anahtar nokta, en yakın komşu eşleştirme\", [\"durum\", \"doğruluk\"], [[\"90° (kesin)\", f\"{float(((d1 @ d2.T).argmax(1) == torch.arange(N)).double().mean()):.2f}\"], [\"30°, yön düzeltmeli\", f\"{acc:.2f}\"], [\"30°, yön düzeltmesiz\", f\"{acc0:.2f}\"]],\n                note=\"Doğru θ verilmezse betimleyici dönmeye karşı korumasızdır: SIFT'in baskın yön adımı bu yüzden var.\")\n    R.test(\"dönme değişmezliği\", rot)\n    if \"describe\" not in ns:\n        return\n    big = kps.repeat(4, 1)\n    t1 = bench(lambda: ns[\"describe\"](img, big), repeat=3)\n    t2 = bench(lambda: [ns[\"describe\"](img, big[i:i + 1]) for i in range(100)], repeat=1, number=1) * len(big) / 100\n    R.table(f\"{len(big)} anahtar nokta\", [\"yöntem\", \"süre\"], [[\"tek seferde\", fmt_time(t1)], [\"nokta başına çağrı (tahmini)\", fmt_time(t2)]])\n",
 "x1": "import numpy as np\nimport cv2\nfrom skimage import data\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    img = data.camera()\n    ii = cv2.integral(img).astype(np.float64)\n    im = img.astype(np.int64)\n\n    def bx():\n        close(need(ns, \"box_sum\")(ii, 10, 20, 7, 9), im[10:17, 20:29].sum(), what=\"kutu toplamı\")\n        dx, dy = need(ns, \"haar\")(ii, 100, 200, 3)\n        close(dx, im[97:103, 200:203].sum() - im[97:103, 197:200].sum(), what=\"dx\")\n        close(dy, im[100:103, 197:203].sum() - im[97:100, 197:203].sum(), what=\"dy\")\n    R.test(\"box_sum ve haar\", bx)\n\n    rng = np.random.default_rng(0)\n    pts = np.stack([rng.integers(60, 450, 200), rng.integers(60, 450, 200)], 1)\n\n    def inv():\n        d1 = need(ns, \"usurf\")(img, pts, 2)\n        ensure(d1.shape == (200, 64), \"(N, 64)\")\n        img2 = np.clip(img.astype(float) * 0.6 + 40, 0, 255).astype(np.uint8)\n        d2 = ns[\"usurf\"](img2, pts, 2)\n        cos = (d1 * d2).sum(1)\n        acc = ((d1 @ d2.T).argmax(1) == np.arange(200)).mean()\n        raw = lambda im: np.stack([im[y - 20:y + 20, x - 20:x + 20].astype(float).ravel() for y, x in pts])\n        r1, r2 = raw(img), raw(img2)\n        accr = (np.argmin(((r1[:, None] - r2[None]) ** 2).sum(-1), 1) == np.arange(200)).mean()\n        ensure(np.median(cos) > 0.98 and acc > accr + 0.2, f\"medyan kosinüs {np.median(cos):.3f}, doğruluk {acc:.2f} / ham {accr:.2f}\")\n        R.table(\"Parlaklık ve kontrast değişimi (0.6·I + 40, uint8'e yuvarlanmış)\", [\"betimleyici\", \"eşleşme doğruluğu\"], [[\"U-SURF (64)\", f\"{acc:.2f}\"], [\"ham 40 × 40 yama, SSD\", f\"{accr:.2f}\"]],\n                note=f\"Kosinüs medyanı {np.median(cos):.3f}, en küçüğü {cos.min():.3f}: düz bölgelerde Haar yanıtları küçük olduğu için uint8 yuvarlama gürültüsü baskınlaşır.\")\n    R.test(\"aydınlatma değişmezliği\", inv)\n",
 "x2": "import numpy as np\nimport cv2\nfrom labcheck import need, ensure\n\n\ndef check(ns, R):\n    def t():\n        Xtr, ytr = need(ns, \"shapes\")(400, 0)\n        Xte, yte = ns[\"shapes\"](200, 1)\n        ensure(Xtr.shape == (400, 64, 64) and set(np.unique(ytr)) == {0, 1}, \"veri\")\n        F = need(ns, \"hog_features\")(Xtr[:3])\n        ensure(F.shape == (3, 1764), f\"64 × 64 için 7 × 7 × 4 × 9 = 1764 boyut, {F.shape}\")\n        h, r = need(ns, \"train_eval\")(Xtr, ytr, Xte, yte)\n        ensure(h > 0.9 and h > r + 0.15, f\"HOG {h:.2f}, ham {r:.2f}\")\n        R.table(\"Daire mi kare mi? (rastgele konum, boyut, parlaklık)\", [\"öznitelik\", \"boyut\", \"test doğruluğu\"], [[\"ham pikseller\", 4096, f\"{r:.2f}\"], [\"HOG\", 1764, f\"{h:.2f}\"]],\n                note=\"Yaya dedektörünün 64 × 128 penceresi aynı ayarlarla 3780 boyut verir (OpenCV HOGDescriptor varsayılanı).\")\n    R.test(\"HOG + doğrusal SVM\", t)\n",
 "x3": "import numpy as np\nimport cv2\nfrom skimage import data\nfrom labcheck import need, ensure\n\n\ndef check(ns, R):\n    def mr():\n        rng = np.random.default_rng(0)\n        a = rng.normal(size=(50, 16)); b = np.concatenate([a + 0.01 * rng.normal(size=a.shape), rng.normal(size=(30, 16))])\n        m = need(ns, \"match_ratio\")(a, b)\n        ensure(len(m) == 50 and all(i == j for i, j in m), \"gürültülü kopyalar eşleşmeli\")\n        ab = rng.integers(0, 256, (20, 32), dtype=np.uint8)\n        mb = ns[\"match_ratio\"](ab, ab.copy())\n        ensure(len(mb) == 20 and all(i == j for i, j in mb), \"ikili betimleyici (Hamming)\")\n    R.test(\"match_ratio\", mr)\n\n    def ev():\n        img = data.camera()\n        rows = []\n        for ang, sc in [(0, 1.0), (30, 1.0), (45, 0.7), (90, 1.3)]:\n            M = cv2.getRotationMatrix2D((256, 256), ang, sc)\n            s = need(ns, \"evaluate\")(img, M, \"sift\"); o = ns[\"evaluate\"](img, M, \"orb\")\n            rows.append([f\"{ang}°, ×{sc}\", f\"{s[1]}/{s[0]}\", f\"{o[1]}/{o[0]}\"])\n            ensure(s[0] > 0 and s[1] / s[0] > 0.7, f\"SIFT doğru oranı düşük ({ang}°, ×{sc})\")\n        R.table(\"Doğru / toplam eşleşme (oran testi 0.8)\", [\"dönüşüm\", \"SIFT\", \"ORB\"], rows,\n                note=\"Alıştırma 6'nın panoraması bu eşleşmelerden RANSAC ile homografi kestirir (M9.15).\")\n    R.test(\"SIFT ve ORB eşleştirme\", ev)\n"
}

In [ ]:
import importlib, labcheck, checks_m9_14_features as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

**Neden?** Görüntüler arasında nokta eşleştirmek (kayıt, panorama, 3-B) için ölçek, dönme, aydınlatma ve bakış açısı değişse de **aynı** noktaları bulan ve aynı biçimde **betimleyen** öznitelikler gerekir.

**Ölçek uzayı:** $L(x,y,\sigma)=G(x,y,\sigma)*f(x,y)$. SIFT ölçeği **oktavlara** böler (her oktav bir öncekinin yarı çözünürlüğü), her oktavı $s$ aralığa: $k=2^{1/s}$, oktav başına $s+3$ Gauss görüntüsü ($s+2$ DoG). Yeni oktavın ilk görüntüsü, öncekinin $2\sigma$'lık görüntüsünün ikiye alt örneklenmesidir.

**Anahtar noktalar:** $D(x,y,\sigma)=L(x,y,k\sigma)-L(x,y,\sigma)\approx(k-1)\sigma^2\nabla^2G*f$ (ölçek-normalize LoG). Her örnek, kendi ölçeğindeki 8 ve üstteki/alttaki ölçeklerdeki 9'ar komşusuyla (26) karşılaştırılır. Düşük kontrastlılar ($|D|<0.03$) ve kenar yanıtları ($\mathrm{Tr}^2(H)/\mathrm{Det}(H)\ge(r+1)^2/r$, $r=10$; Harris'teki fikir) atılır.

**Yön ve betimleyici:** anahtar noktanın ölçeğinde gradyanlardan 36 bölmeli yön histogramı, tepe = baskın yön. Betimleyici bu yöne döndürülmüş $16\times16$ örnekten: $4\times4$ alt bölge × 8 yön = **128** boyut; L2 normalize, 0.2'de kırp, tekrar normalize (doğrusal olmayan aydınlatmaya dayanıklılık).

**SURF:** Hessian $H=\begin{bmatrix}L_{xx}&L_{xy}\\L_{xy}&L_{yy}\end{bmatrix}$'nin determinantı; Gauss ikinci türevleri **kutu süzgeçlerle** ($9\times9$ ↔ $\sigma=1.2$) yaklaşıklanır, **integral görüntüyle** her boyutta sabit sürede: $\det\approx D_{xx}D_{yy}-(0.9D_{xy})^2$ (yanıtlar süzgeç alanına bölünür). Ölçek uzayı görüntüyü küçültmeden **süzgeci büyüterek** (9, 15, 21, 27, …) kurulur. Betimleyici: $20s$'lik pencerede $4\times4$ alt bölge, her birinde Haar yanıtlarından $(\sum d_x,\sum|d_x|,\sum d_y,\sum|d_y|)$ → 64 boyut.

**HOG:** merkezi fark gradyanları, $8\times8$ hücrede büyüklükle ağırlıklı 9 yönlü (0–180°) histogram, $2\times2$ hücrelik örtüşen bloklarda L2-Hys normalizasyonu, hepsini birleştir. 64 × 128 pencere: $7\times15$ blok × 4 hücre × 9 = 3780.

## 1 · Elle

$\mathrm{Tr}^2/\mathrm{Det}=22^2/39=12.41\ge12.1$: eğrilikler çok farklı, nokta bir kenarın üzerinde.

HOG: yatayda $64/8-1=7$, dikeyde $128/8-1=15$ blok.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**SIFT ölçek uzayı, $s=2$, $\sigma_1=\sqrt2/2$**

- `k`: $k=2^{1/s}$
- `n`: Oktav başına Gauss görüntüsü ($s+3$)
- `s3`: Oktavdaki 3. görüntünün σ'sı
- `r`: Kenar eşiği $(r+1)^2/r$, $r=10$
- `tr`: $D_{xx}=20$, $D_{yy}=2$, $D_{xy}=1$: $\mathrm{Tr}^2/\mathrm{Det}$ (≥ eşik → kenar, at)

**SURF ve HOG**

- `det`: SURF: $D_{xx}=5$, $D_{yy}=4$, $D_{xy}=2$, $w=0.9$: $\det$
- `sg`: SURF: 27 × 27 süzgecin karşılığı σ (9 × 9 ↔ 1.2)
- `hog`: HOG boyutu: 64 × 128 pencere, 8 × 8 hücre, 2 × 2 blok (adım 1 hücre), 9 yön
- `bin`: HOG: 100°'lik gradyanın bölmesi (9 bölme, 0–180°, 20°'lik)

In [ ]:
cevaplar = dict(
    k=None,
    n=None,
    s3=None,
    r=None,
    tr=None,
    det=None,
    sg=None,
    hog=None,
    bin=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"k":[1.414214,0.0001],"n":[5,0],"s3":[1.414214,0.0001],"r":[12.1,0],"tr":[12.41026,0.001],"det":[16.76,0.0001],"sg":[3.6,0.0001],"hog":[3780,0],"bin":[5,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python, tek bir 16 × 16 görüntü (4 hücre, 1 blok) için HOG — `skimage.feature.hog(img, 9, (8, 8), (2, 2), block_norm="L2-Hys")` ile aynı.

- `gradients(img)` → `(gx, gy)`: merkezi fark $f(y,x+1)-f(y,x-1)$ ve $f(y+1,x)-f(y-1,x)$, görüntü kenarındaki satır/sütunlarda 0.
- `cell_histogram(gx, gy, y0, x0, cell=8, nbins=9)` — hücredeki her pikselin büyüklüğü $\sqrt{g_x^2+g_y^2}$, açısı $\mathrm{atan2}(g_y,g_x)$ derece mod 180, bölme $\min(\lfloor a/(180/n)\rfloor,n-1)$; toplamı hücre alanına böl.
- `l2hys(v, eps=1e-5)` — $v/\sqrt{\lVert v\rVert^2+\varepsilon^2}$, 0.2'de kırp, yine normalize.
- `hog16(img)` — 4 hücrenin histogramlarını (satır sırası) birleştirip `l2hys`; 36 boyut.

In [ ]:
%%writefile stage_b.py
import math


def gradients(img):
    ...


def cell_histogram(gx, gy, y0, x0, cell=8, nbins=9):
    ...


def l2hys(v, eps=1e-5):
    ...


def hog16(img):
    ...


img = [[(x * y) % 17 for x in range(16)] for y in range(16)]
print(len(hog16(img)))

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy, **döngü yok**.

**`hog(img, nbins=9, cell=8, block=2)`** — `skimage.feature.hog(img, nbins, (cell, cell), (block, block), block_norm="L2-Hys", feature_vector=True)` ile aynı (b'deki tanım, tüm görüntü; artakalan kenar pikselleri atılır; bloklar 1 hücre adımla, sıra `(blok_y, blok_x, hücre_y, hücre_x, yön)`). Hücre histogramları için ofsetli `np.bincount`; bloklar için `sliding_window_view`.

**`fast_hessian(img, L=9)`** — SURF det yanıtı, $L=9+6k$, lob $l=L/3$:

- $D_{yy}$: üst üste üç $l\times(2l-1)$ şerit, ağırlıklar $+1,-2,+1$ (merkezli, $L\times(2l-1)$); $D_{xx}$ devriği.
- $D_{xy}$: merkezin çevresinde 1 piksellik boşlukla dört $l\times l$ kare; sol üst ve sağ alt $+1$, sağ üst ve sol alt $-1$.
- Kutu toplamları **integral görüntüden** (sıfır dolgulu; görüntü dışı 0); her yanıt $L^2$'ye bölünür; $\det=D_{xx}D_{yy}-(0.9D_{xy})^2$.

In [ ]:
%%writefile stage_c.py
import numpy as np


def hog(img, nbins=9, cell=8, block=2):
    ...


def fast_hessian(img, L=9):
    ...


from skimage import data
print(hog(data.camera()[:128, :64]).shape)

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

DoG piramidi ve **ölçek seçimi**.

- `dog_pyramid(img, octaves=4, s=3, sigma0=1.6)` — `img (H, W)` `float64` tensör. Her oktavda $s+3$ Gauss görüntüsü: ilki tabanın $\sigma_0$ ile bulanığı (ilk oktavda görüntünün kendisi), sonrakiler artımlı $\sqrt{\sigma_i^2-\sigma_{i-1}^2}$, $\sigma_i=\sigma_0k^i$. Gauss: yarıçap $\lceil4\sigma\rceil$, toplamı 1, `replicate`, ayrılabilir `conv2d`. DoG: `(s+2, H_o, W_o)` tensör. Sonraki oktavın tabanı: bu oktavın $s$. görüntüsü `[::2, ::2]`. Liste döndür.
- `detect(img, octaves=4, s=3, sigma0=1.6, thr=0.03)` — her oktavda 3 × 3 × 3 yerel en büyük ya da en küçük (`max_pool3d`, eşitlik dahil), $|D|>$ `thr`, ilk ve son DoG katmanı ve 1 piksellik kenar hariç. Anahtar nokta `(x·2^o, y·2^o, σ_0·2^{o+i/s}, D)` listesi.

Kontrol, yarıçapları farklı Gauss bloblarında bulunan ölçeğin blob boyutuyla **orantılı** çıktığını ve DoG yanıtının ölçekten bağımsız kaldığını gösterir.

In [ ]:
%%writefile stage_d.py
import math
import torch
import torch.nn.functional as F


def dog_pyramid(img, octaves=4, s=3, sigma0=1.6):
    ...


def detect(img, octaves=4, s=3, sigma0=1.6, thr=0.03):
    ...


x = torch.zeros(64, 64, dtype=torch.float64); x[28:36, 28:36] = 1
print(len(detect(x, octaves=2)))

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

`describe(img, kps)` — `img (H, W)` NumPy, `kps (N, 4)` `float64` tensör `[x, y, σ, θ]` → `(N, 128)`; tüm noktalar **aynı anda** (döngü yok):

1. Örnek ızgarası $u,v\in\{-7.5,…,7.5\}$ (16 × 16), adım $\Delta=3\sigma/4$, θ ile döndürülmüş: $p=(x,y)+\Delta R(\theta)(u,v)$.
2. Döndürülmüş çerçevede gradyan: $g_u=I(p+\Delta Re_u)-I(p-\Delta Re_u)$, $g_v$ benzer — `grid_sample` (bilineer, `align_corners=True`, `padding_mode="border"`), dört örnekleme.
3. Büyüklük × $e^{-(u^2+v^2)/(2\cdot8^2)}$, açı `atan2(gv, gu) mod 2π`, 8 bölme (sert, `floor`).
4. Alt bölge $(\lfloor(v+7.5)/4\rfloor,\lfloor(u+7.5)/4\rfloor)$, indis `alt_bölge·8 + bölme`, `scatter_add_`.
5. L2 normalize, 0.2'de kırp, tekrar normalize.

Gradyanlar anahtar noktanın kendi çerçevesinde hesaplandığı için betimleyici **dönmeden bağımsızdır**.

In [ ]:
%%writefile stage_e.py
import math
import numpy as np
import torch
import torch.nn.functional as F


def describe(img, kps):
    ...


from skimage import data
kps = torch.tensor([[256.0, 256.0, 2.0, 0.3]], dtype=torch.float64)
print(describe(data.camera(), kps).shape)

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · U-SURF betimleyicisi: Haar dalgacıkları ve integral görüntü

U-SURF (yönsüz SURF) betimleyicisi, integral görüntüyle:

- `box_sum(ii, y0, x0, h, w)` — integral görüntüden (`(H+1, W+1)`, `cv2.integral` biçimi) dört erişimle toplam; dizi argümanlarıyla vektörel çalışsın.
- `haar(ii, y, x, s)` — $(y,x)$ merkezli $2s\times2s$ Haar: $d_x$ = sağ yarı − sol yarı, $d_y$ = alt yarı − üst yarı.
- `usurf(img, pts, s)` — her nokta için $20s$'lik pencerede $4\times4$ alt bölge × $5\times5$ örnek (adım $s$, örnek merkezleri $(i-9.5)s$, $i=0..19$), Gauss ağırlığı σ = 3.3s, alt bölge başına $(\sum d_x,\sum|d_x|,\sum d_y,\sum|d_y|)$; 64 boyut, L2 normalize. `s` tamsayı.

In [ ]:
%%writefile extra_x1.py
import numpy as np
import cv2


def box_sum(ii, y0, x0, h, w):
    ...


def haar(ii, y, x, s):
    ...


def usurf(img, pts, s):
    ...


from skimage import data
print(usurf(data.camera(), np.array([[256, 256]]), 2).shape)

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · HOG + doğrusal SVM ile sınıflandırma

- `shapes(n, seed)` — `(X, y)`: 64 × 64 görüntüler, sınıf 0 rastgele konum/boyutta dolu daire, sınıf 1 dolu kare; rastgele parlaklık (arka plan ve nesne), Gauss gürültüsü σ = 10.
- `hog_features(X)` — her görüntünün `skimage.feature.hog` vektörü (9 yön, 8 × 8, 2 × 2).
- `train_eval(Xtr, ytr, Xte, yte)` → `(acc_hog, acc_raw)`: `sklearn.svm.LinearSVC(C=0.01)` ile HOG ve ham pikseller (0–1'e ölçekli) doğrulukları.

In [ ]:
%%writefile extra_x2.py
import numpy as np
import cv2
from skimage.feature import hog
from sklearn.svm import LinearSVC


def shapes(n, seed):
    ...


def hog_features(X):
    ...


def train_eval(Xtr, ytr, Xte, yte):
    ...


X, y = shapes(10, 0)
print(X.shape, hog_features(X).shape)

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · OpenCV SIFT ve ORB ile eşleştirme, Lowe oran testi

- `match_ratio(d1, d2, ratio=0.8, cross=False)` — her $d_1$ için iki en yakın komşu (L2; ikili betimleyicide Hamming); en yakın / ikinci < `ratio` ise eşleşme `(i, j)` listesi. `cross=True` ise karşılıklı en yakın olma şartı da.
- `evaluate(img, M, detector)` — `M` 2 × 3 afin; ikinci görüntü `cv2.warpAffine`; `detector` `"sift"` (`cv2.SIFT_create()`) ya da `"orb"` (`cv2.ORB_create(2000)`); oran testiyle eşleşmeler; birinci noktayı `M` ile taşıyınca ikinciye 3 pikselden yakın olanlar doğru. `(eşleşme, doğru)` döndür.

In [ ]:
%%writefile extra_x3.py
import numpy as np
import cv2


def match_ratio(d1, d2, ratio=0.8, cross=False):
    ...


def evaluate(img, M, detector):
    ...


from skimage import data
print(evaluate(data.camera(), cv2.getRotationMatrix2D((256, 256), 20, 0.8), "sift"))

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: SURF'ü sıfırdan uygula

[Bay, Tuytelaars, Van Gool — SURF: Speeded Up Robust Features (ECCV 2006)](https://doi.org/10.1007/11744023_32)

Bu modüldeki Fast-Hessian (c) ve U-SURF betimleyicisini (x1) tam SURF'e tamamla: 4 oktav × 4 ölçek, 3 × 3 × 3 non-maximum suppression, ölçek ve konum interpolasyonu (2. derece Taylor), 60°'lik kayan pencereyle baskın yön, döndürülmüş 64 boyutlu betimleyici ve Laplacian işaretiyle hızlı eşleştirme.


**Adımlar**

1. Oktav ve ölçekler: süzgeç boyutları 9, 15, 21, 27 / 15, 27, 39, 51 / … ve örnekleme adımları.
2. Interpolasyon: $\hat x=-\left(\frac{\partial^2H}{\partial x^2}\right)^{-1}\frac{\partial H}{\partial x}$, sonlu farklarla.
3. Yön: 6s yarıçapında 4s'lik Haar yanıtları, σ = 2s Gauss ağırlığı, π/3'lük kayan pencere.
4. Oxford affine veri setinde (graf, wall, boat, bikes…) tekrarlanabilirlik ve eşleşme skoru; OpenCV SIFT ile karşılaştır.

**Kabul ölçütleri**

- Rotasyon (boat) ve ölçek dizilerinde SIFT'e yakın tekrarlanabilirlik (±10 puan).
- Saniyede işlenen piksel olarak SIFT'ten hızlı (aynı dilde, ör. NumPy/Torch sürümleri).
- Laplacian işareti ön elemesinin eşleştirme süresine etkisi.

## Makale ödevi: Dalal–Triggs yaya dedektörü

[Dalal & Triggs — Histograms of Oriented Gradients for Human Detection (CVPR 2005)](https://doi.org/10.1109/CVPR.2005.177)

INRIA Person (ya da Penn-Fudan) verisinde 64 × 128 pencerelerden HOG + doğrusal SVM yaya dedektörü eğit; zor negatif madenciliği (hard negative mining), çok ölçekli kayan pencere ve non-maximum suppression (M4.9) ekle; makaledeki DET eğrisini (miss rate – FPPW) üret.


**Adımlar**

1. c aşamasındaki HOG'u kullan; makaledeki iyileştirmeleri ekle: üçgen (bilineer) oy dağıtımı ve blok Gauss penceresi (σ = 8).
2. Doğrusal SVM (C = 0.01); negatiflerden rastgele 10 pencere / görüntü, sonra zor negatifler.
3. Görüntü piramidi (ölçek 1.05) ve kutu NMS.
4. Ablasyon: hücre boyutu (6, 8, 10), yön sayısı (6, 9, 18), işaretsiz / işaretli yön.

**Kabul ölçütleri**

- 10⁻⁴ FPPW'de miss rate ≤ %20 (makale ≈ %11).
- Zor negatif madenciliğinin etkisini gösteren iki DET eğrisi.
- `cv2.HOGDescriptor_getDefaultPeopleDetector` ile nitel karşılaştırma.

## Makale ödevi: SIFT dedektör + betimleyici (uzantı)

[Lowe — Distinctive Image Features from Scale-Invariant Keypoints (IJCV 2004)](https://doi.org/10.1023/B:VISI.0000029664.99615.94)

d (DoG piramidi) ve e (betimleyici) aşamalarını tam SIFT'e tamamla: başlangıçta 2× büyütme, alt piksel ve alt ölçek interpolasyonu, kontrast ve kenar eleme, 36 bölmeli yön histogramı (parabol uydurma, %80'in üstündeki ikinci tepeler için ek anahtar nokta), trilineer oy dağıtımı. `cv2.SIFT_create` ile anahtar nokta konumlarını ve eşleşme doğruluğunu karşılaştır.


**Adımlar**

1. Lowe'un parametreleri: σ = 1.6, s = 3, kontrast 0.03, r = 10.
2. Yön ve betimleyiciyi Gauss pencereli ve trilineer olarak yaz.

**Kabul ölçütleri**

- OpenCV SIFT anahtar noktalarının %70'inden fazlasıyla 1.5 piksel içinde örtüşme.
- Döndürülmüş + ölçeklenmiş görüntü çiftinde OpenCV SIFT'e yakın doğru eşleşme oranı.

## Kaynaklar

- [Lowe — Distinctive Image Features from Scale-Invariant Keypoints (IJCV 2004)](https://doi.org/10.1023/B:VISI.0000029664.99615.94) — ©
- [Bay, Tuytelaars, Van Gool — SURF: Speeded Up Robust Features (ECCV 2006)](https://doi.org/10.1007/11744023_32) — ©
- [Dalal & Triggs — Histograms of Oriented Gradients for Human Detection (CVPR 2005)](https://doi.org/10.1109/CVPR.2005.177) — ©
- [Gonzalez & Woods — Digital Image Processing, 4. baskı, Bölüm 11.7](https://www.imageprocessingplace.com/) — ©
- [scikit-image — feature.hog](https://scikit-image.org/docs/stable/api/skimage.feature.html#skimage.feature.hog) — BSD-3